In [1]:
!pip install timm

In [2]:
!pip install opencv-contrib-python

In [3]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm
import gc
import matplotlib.pyplot as plt
import numpy as np
import os
import random
import time
import timm # WICHTIG: pip install timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms.functional as TF  # Das fixiert den NameError
import re
import cv2 # Für das Resizing der Disparity-Map
from torch.cuda.amp import GradScaler, autocast





# Device Selection (GPU/CPU)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Training auf GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("Training auf CPU (Langsam!)")



/home/slarc/miniconda3/envs/stereo_wsl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Training auf GPU: NVIDIA GeForce RTX 3080 Ti


In [4]:
# --- DATASET V9 (Mit Asymmetric Augmentation) ---
class StereoDataset(Dataset):
    def __init__(self, data_dir, mode='train', use_crop=True, use_augmentation=True):
        self.data_dir = data_dir
        self.mode = mode
        self.use_crop = use_crop
        self.use_augmentation = use_augmentation
        
        split = 'train' if mode == 'train' else 'val'
        
        # Pfadsuche
        self.img_root = os.path.join(data_dir, 'FlyingThings3D_subset_image_clean', 'FlyingThings3D_subset', split, 'image_clean')
        if not os.path.exists(self.img_root):
            self.img_root = os.path.join(data_dir, split, 'image_clean')
            
        if not os.path.exists(self.img_root):
             raise ValueError(f"❌ Bild-Ordner nicht gefunden:\n{self.img_root}")

        print(f"[{mode.upper()}] Scanne Bilder in: {self.img_root}")

        self.left_files = []
        self.right_files = []
        self.disp_left_files = []
        self.disp_right_files = []
        
        for root, dirs, files in os.walk(self.img_root):
            for file in files:
                if file.endswith('.png') and 'left' in root:
                    l_path = os.path.join(root, file)
                    r_path = l_path.replace('left', 'right')
                    dl_path = l_path.replace('image_clean', 'disparity').replace('.png', '.pfm')
                    dr_path = r_path.replace('image_clean', 'disparity').replace('.png', '.pfm')
                    
                    if os.path.exists(dl_path):
                        self.left_files.append(l_path)
                        self.right_files.append(r_path)
                        self.disp_left_files.append(dl_path)
                        self.disp_right_files.append(dr_path)
                        
        print(f"[{mode.upper()}] {len(self.left_files)} Paare gefunden.")

    def load_pfm(self, file):
        if not os.path.exists(file): return np.zeros((480, 640), dtype=np.float32)
        with open(file, "rb") as f:
            header = f.readline().decode('utf-8').rstrip()
            if header == 'PF': color = True
            elif header == 'Pf': color = False
            else: raise Exception('Keine PFM Datei.')

            dims = f.readline().decode('utf-8').split()
            width = int(dims[0])
            height = int(dims[1])

            scale = float(f.readline().decode('utf-8').rstrip())
            if scale < 0:
                endian = '<'
                scale = -scale
            else:
                endian = '>'

            data = np.fromfile(f, endian + 'f')
            shape = (height, width, 3) if color else (height, width)

            data = np.reshape(data, shape)
            data = np.flipud(data)
            
            # --- FIX ---
            # 1. NaN/Inf bereinigen
            data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)
            
            # 2. Absolutwert nehmen! 
            # Deine Daten sind negativ (-79 bis -1), das muss positiv werden.
            data = np.abs(data)
            # -----------

            return data.copy()

    def __len__(self):
        return len(self.left_files)

    def __getitem__(self, idx):
        l_path = self.left_files[idx]
        r_path = self.right_files[idx]
        
        left = Image.open(l_path).convert('L')
        right = Image.open(r_path).convert('L')
        dl = self.load_pfm(self.disp_left_files[idx])
        dr = self.load_pfm(self.disp_right_files[idx])
        
        orig_w, orig_h = left.size
        target_w, target_h = 640, 480
        
        left = left.resize((target_w, target_h), Image.BILINEAR)
        right = right.resize((target_w, target_h), Image.BILINEAR)
        scale_x = target_w / orig_w
        dl = cv2.resize(dl, (target_w, target_h), interpolation=cv2.INTER_LINEAR) * scale_x
        dr = cv2.resize(dr, (target_w, target_h), interpolation=cv2.INTER_LINEAR) * scale_x

        l_np = np.array(left, dtype=np.float32) / 255.0
        r_np = np.array(right, dtype=np.float32) / 255.0
        dl_np = np.ascontiguousarray(dl, dtype=np.float32)
        dr_np = np.ascontiguousarray(dr, dtype=np.float32)

        if self.mode == 'train' and self.use_crop:
            crop_h, crop_w = 320, 640
            y = random.randint(0, target_h - crop_h)
            x = random.randint(0, target_w - crop_w)
            
            l_np = l_np[y:y+crop_h, x:x+crop_w]
            r_np = r_np[y:y+crop_h, x:x+crop_w]
            dl_np = dl_np[y:y+crop_h, x:x+crop_w]
            dr_np = dr_np[y:y+crop_h, x:x+crop_w]
            
            if self.use_augmentation:
                def augment_photo(img):
                    mult = 0.8 + np.random.rand() * 0.4 
                    img = img * mult
                    mean = img.mean()
                    contrast = 0.8 + np.random.rand() * 0.4
                    img = (img - mean) * contrast + mean
                    img = np.clip(img, 0, 1)
                    gamma = 0.8 + np.random.rand() * 0.4
                    img = img ** gamma
                    return np.clip(img, 0, 1)

                l_np = augment_photo(l_np)
                r_np = augment_photo(r_np)

        l_t = torch.from_numpy(l_np).unsqueeze(0)
        r_t = torch.from_numpy(r_np).unsqueeze(0)
        dl_t = torch.from_numpy(dl_np).unsqueeze(0)
        dr_t = torch.from_numpy(dr_np).unsqueeze(0)
        
        return l_t, r_t, dl_t, dr_t

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

# --- Helper Classes (Standard) ---
class Conv2dReLU6(nn.Module):
    def __init__(self, in_c, out_c, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.conv = nn.Conv2d(in_c, out_c, kernel_size, stride, padding, bias=False)
        self.bn = nn.BatchNorm2d(out_c)
        self.act = nn.ReLU6(inplace=True)
    def forward(self, x): return self.act(self.bn(self.conv(x)))

class DepthwiseSeparable(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.depthwise = nn.Conv2d(in_c, in_c, 3, padding=1, groups=in_c, bias=False)
        self.pointwise = nn.Conv2d(in_c, out_c, 1, bias=False)
        self.bn = nn.BatchNorm2d(out_c)
        self.act = nn.ReLU6(inplace=True)
    def forward(self, x): return self.act(self.bn(self.pointwise(x)))

class StructureBlock(nn.Module):
    def __init__(self):
        super().__init__()
        sobel_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]).view(1, 1, 3, 3)
        sobel_y = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]]).view(1, 1, 3, 3)
        self.register_buffer('k_sx', sobel_x)
        self.register_buffer('k_sy', sobel_y)
    def forward(self, x):
        sx = F.conv2d(x, self.k_sx, padding=1)
        sy = F.conv2d(x, self.k_sy, padding=1)
        return torch.abs(sx) + torch.abs(sy)

# --- NPU Friendly Occlusion Head ---
class NPUOcclusionHead(nn.Module):
    def __init__(self):
        super().__init__()
        # Input: Disp(1) + Structure(1) + Confidence(1) = 3 Channels
        # Wir verzichten auf Warping-Inputs!
        self.conv1 = Conv2dReLU6(3, 32, 3, 1, 1)
        self.conv2 = Conv2dReLU6(32, 16, 3, 1, 1)
        self.out = nn.Conv2d(16, 1, 1)
        
        nn.init.zeros_(self.out.bias) # Bias 0 -> Sigmoid 0.5 start
    
    def forward(self, disp, structure, confidence):
        # Das Netz muss lernen: 
        # Hohe Kanten + Tiefe Confidence + Disparitätssprung = Okklusion
        x = torch.cat([disp, structure, confidence], dim=1)
        x = self.conv1(x)
        x = self.conv2(x)
        return self.out(x) # Logits

class MiniUNetRefiner(nn.Module):
    def __init__(self, in_channels, max_disp=192, edge_in_channels=2):
        super().__init__()
        self.max_disp = max_disp
        
        # Reduced channels for Speed (32/64 -> 24/48 is often enough)
        # Keeping it conservative here: 32 base
        self.enc1 = Conv2dReLU6(in_channels, 32)
        self.down1 = Conv2dReLU6(32, 32, stride=2)
        
        self.enc2 = Conv2dReLU6(32, 48) # Reduced from 64
        self.down2 = Conv2dReLU6(48, 48, stride=2) # Reduced from 64
        
        # Center: Lighter Depthwise
        self.center = nn.Sequential(
            Conv2dReLU6(48, 48), 
            DepthwiseSeparable(48, 48)
        )
        
        # Upsampling fixed to align_corners=False
        self.up2 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.dec2 = Conv2dReLU6(48 + 48, 32) # Input: Center + Enc2
        
        self.up1 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.dec1 = Conv2dReLU6(32 + 32, 16) # Input: Dec2 + Enc1
        
        self.final = nn.Conv2d(16, 1, kernel_size=3, padding=1)
        
        self.edge_refine = nn.Sequential(
            nn.Conv2d(edge_in_channels, 16, 3, padding=1), 
            nn.ReLU6(inplace=True),
            nn.Conv2d(16, 1, 3, padding=1)
        )
        
        nn.init.uniform_(self.final.weight, -0.01, 0.01)
        nn.init.uniform_(self.edge_refine[-1].weight, -0.001, 0.001)

    def forward(self, disp_curr, features, structure):
        disp_norm = disp_curr / self.max_disp
        x = torch.cat([disp_norm, features, structure], dim=1)
        
        e1 = self.enc1(x)
        e2 = self.enc2(self.down1(e1))
        
        c = self.center(self.down2(e2))
        
        # Concatenation sizes must match new channel counts
        d2 = self.dec2(torch.cat([self.up2(c), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        
        res_main = self.final(d1)
        res_edge = self.edge_refine(torch.cat([disp_norm, structure], dim=1))
        
        if disp_curr.shape[1] > 1: disp_base = disp_curr[:, 0:1, :, :]
        else: disp_base = disp_curr
            
        out = disp_base + 0.7 * (res_main + res_edge)
        return torch.clamp(out, 0, self.max_disp)

# --- HAUPTKLASSE V9.5 (NPU SPEED) ---
class StereoNet_NPU_V9(nn.Module):
    def __init__(self, max_disp=192):
        super().__init__()
        self.max_disp = max_disp
        
        # ÄNDERUNG: Groups von 8 auf 12 erhöht (bessere Accuracy)
        self.groups = 12 
        
        self.softmax_temp = nn.Parameter(torch.tensor(1.0))
        
        # Backbone (Gray)
        self.backbone = timm.create_model('mobilenetv3_large_100', pretrained=True, features_only=True, out_indices=(1, 2, 4), in_chans=1)
        
        # Adapters
        self.lat_s4 = nn.Conv2d(24, 32, 1, bias=False)
        self.lat_s8 = nn.Conv2d(40, 32, 1, bias=False)
        
        # Global Context (wie zuvor besprochen)
        self.global_context = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(40, 32, 1, bias=False),
            nn.ReLU6(inplace=True)
        )

        self.fuse = Conv2dReLU6(32, 32, 3, 1, 1)
        self.feat_norm = nn.GroupNorm(8, 32)
        self.structure = StructureBlock()
        
        # --- ÄNDERUNG: Verbesserte Cost Volume Verarbeitung ---
        
        # 1. Moderater Compute: 2-stufige Projektion (96 -> 32 -> Groups)
        # Statt direkt 96 -> 12. Bringt mehr Kapazität für Feature-Mischung.
        self.cost_reducer = nn.Sequential(
            nn.Conv2d(96, 32, kernel_size=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU6(inplace=True),
            nn.Conv2d(32, self.groups, kernel_size=1, bias=False)
        )
        
        # 2. Accuracy Priority: Pre-Filter für Matching-Konsistenz
        # Ein kleines 3x3 Conv vor dem großen Spatial Filter glättet Rauschen
        self.cost_pre_filter = DepthwiseSeparable(self.groups * (max_disp // 4), self.groups * (max_disp // 4))

        self.spatial_filter = nn.Sequential(
            DepthwiseSeparable(self.groups * (max_disp // 4), self.groups * (max_disp // 4)),
            nn.Conv2d(self.groups * (max_disp // 4), max_disp // 4, kernel_size=1, bias=False)
        )
        
        # Refiners (angepasst an die leichteren Channels der vorherigen Antwort)
        self.refine_low = MiniUNetRefiner(34, max_disp, edge_in_channels=2)
        self.refine_v1  = MiniUNetRefiner(34, max_disp, edge_in_channels=2)
        self.refine_final = MiniUNetRefiner(35, max_disp, edge_in_channels=3)
        self.occ_head = NPUOcclusionHead()

    def build_cost_volume_fast(self, feat_l, feat_r, max_disp_4):
        B, C, H, W = feat_l.shape
        feat_r_padded = F.pad(feat_r, (max_disp_4 - 1, 0, 0, 0))
        feat_r_unfolded = feat_r_padded.unfold(3, W, 1).permute(0, 3, 1, 2, 4)
        feat_l_exp = feat_l.unsqueeze(1).expand(-1, max_disp_4, -1, -1, -1)
        diff = feat_l_exp - feat_r_unfolded
        return torch.cat([feat_l_exp, feat_r_unfolded, diff], dim=2)

    def forward_single(self, left, right):
        # 1. Feature Extraction
        feats_l_all = self.backbone(left)
        feats_r_all = self.backbone(right)
        
        # Left Side
        f4_l = self.lat_s4(feats_l_all[0])
        f8_l = self.lat_s8(feats_l_all[1])
        g_l = self.global_context(feats_l_all[1])
        
        # align_corners=False (WICHTIG für Hailo)
        f8_l_up = F.interpolate(f8_l, scale_factor=2, mode='bilinear', align_corners=False)
        feat_l = self.fuse(f4_l + f8_l_up + g_l)
        
        # Right Side
        f4_r = self.lat_s4(feats_r_all[0])
        f8_r = self.lat_s8(feats_r_all[1])
        g_r = self.global_context(feats_r_all[1])
        
        f8_r_up = F.interpolate(f8_r, scale_factor=2, mode='bilinear', align_corners=False)
        feat_r = self.fuse(f4_r + f8_r_up + g_r)

        # Norm
        feat_l = F.normalize(self.feat_norm(feat_l), dim=1)
        feat_r = F.normalize(self.feat_norm(feat_r), dim=1)
        
        struct_l = self.structure(left)
        
        # 2. Cost Volume
        B, C, H4, W4 = feat_l.shape
        max_disp_4 = self.max_disp // 4
        
        cost_vol = self.build_cost_volume_fast(feat_l, feat_r, max_disp_4)
        
        # Neue 2-stufige Reduktion
        cost_red = self.cost_reducer(cost_vol.reshape(B * max_disp_4, 96, H4, W4))
        cost_red = cost_red.view(B, max_disp_4 * self.groups, H4, W4)
        
        # Neuer Pre-Filter für Konsistenz
        cost_red = self.cost_pre_filter(cost_red)
        
        cost_out = self.spatial_filter(cost_red)
        
        temp = torch.clamp(self.softmax_temp, 0.5, 2.0)
        prob = F.softmax(-cost_out * temp, dim=1)
        
        confidence, _ = torch.max(prob, dim=1, keepdim=True)
        
        d_range = torch.arange(max_disp_4, device=left.device).view(1, -1, 1, 1).float()
        disp_4 = torch.sum(prob * d_range, dim=1, keepdim=True)
        
        # --- Refinement (mit align_corners=False) ---
        struct_4 = F.interpolate(struct_l, scale_factor=0.25, mode='area')
        disp_4_scaled = disp_4 * 4.0
        disp_low_ref = self.refine_low(disp_4_scaled, feat_l, struct_4)
        
        disp_2 = F.interpolate(disp_low_ref, scale_factor=2, mode='bilinear', align_corners=False)
        feat_2 = F.interpolate(feat_l, scale_factor=2, mode='bilinear', align_corners=False)
        struct_2 = F.interpolate(struct_l, scale_factor=0.5, mode='area')
        disp_v1_ref = self.refine_v1(disp_2, feat_2, struct_2)
        
        disp_1 = F.interpolate(disp_v1_ref, scale_factor=2, mode='bilinear', align_corners=False)
        feat_1 = F.interpolate(feat_l, scale_factor=4, mode='bilinear', align_corners=False)
        
        conf_1 = F.interpolate(confidence, scale_factor=4, mode='bilinear', align_corners=False)
        
        occ_logits = self.occ_head(disp_1, struct_l, conf_1)
        occ_prob = torch.sigmoid(occ_logits)
        
        disp_final = self.refine_final(torch.cat([disp_1, occ_prob], dim=1), feat_1, struct_l)
        
        return disp_final, disp_v1_ref, disp_low_ref, occ_logits

    def core(self, left, right):
        return self.forward_single(left, right)

In [6]:
@torch.no_grad()
def validate(model, val_loader, device):
    model.eval()
    
    total_epe = 0.0
    total_loss = 0.0
    valid_batches = 0
    
    # tqdm für Fortschrittsbalken
    pbar = tqdm(val_loader, desc="🔍 Validierung", leave=False, ncols=150)
    
    # FIX: Jetzt 4 Werte entpacken statt 3
    for left, right, gt_L, gt_R in pbar:
        left, right = left.to(device), right.to(device)
        gt_L = gt_L.to(device)
        # gt_R brauchen wir für EPE-Validierung eigentlich nicht zwingend, 
        # aber wir müssen es entpacken, damit Python nicht meckert.

        # Forward Pass (Wir nutzen nur LR Core für Speed)
        # model.core gibt zurück: (disp_final, disp_v1, disp_low, occ)
        out = model.core(left, right)
        disp_pred = out[0] # Wir nehmen nur die finale Disparität
        
        # Validitäts-Maske (Nur Pixel prüfen, die Ground Truth haben)
        mask = (gt_L > 0) & (gt_L < 192)
        
        if mask.sum() > 0:
            # 1. EPE (End Point Error) berechnen
            # Absoluter Abstand in Pixeln
            diff = torch.abs(disp_pred[mask] - gt_L[mask])
            epe = diff.mean().item()
            
            # 2. Loss berechnen (Smooth L1 als Referenz)
            loss = F.smooth_l1_loss(disp_pred[mask], gt_L[mask], beta=1.0).item()
            
            total_epe += epe
            total_loss += loss
            valid_batches += 1
            
            pbar.set_postfix({'val_epe': f"{epe:.2f}"})

    if valid_batches == 0:
        return 0.0, 0.0

    return (total_loss / valid_batches), (total_epe / valid_batches)


In [7]:
# --- LOSSES V9.3 (Edge-Aware & Adaptive) ---

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# --- 1. SSIM (Benötigt für den Loss, muss definiert sein) ---
def get_ssim_window(window_size, channel):
    def gaussian(window_size, sigma):
        gauss = torch.Tensor([np.exp(-(x - window_size//2)**2/float(2*sigma**2)) for x in range(window_size)])
        return gauss/gauss.sum()
    _1D_window = gaussian(window_size, 1.5).unsqueeze(1)
    _2D_window = _1D_window.mm(_1D_window.t()).float().unsqueeze(0).unsqueeze(0)
    window = _2D_window.expand(channel, 1, window_size, window_size).contiguous()
    return window

class SSIM(nn.Module):
    def __init__(self, window_size=11, channel=1):
        super(SSIM, self).__init__()
        self.window_size = window_size
        self.channel = channel
        self.register_buffer('window', get_ssim_window(window_size, channel))
    def forward(self, img1, img2):
        mu1 = F.conv2d(img1, self.window, padding=self.window_size//2, groups=self.channel)
        mu2 = F.conv2d(img2, self.window, padding=self.window_size//2, groups=self.channel)
        mu1_sq, mu2_sq, mu1_mu2 = mu1.pow(2), mu2.pow(2), mu1*mu2
        sigma1_sq = F.conv2d(img1*img1, self.window, padding=self.window_size//2, groups=self.channel) - mu1_sq
        sigma2_sq = F.conv2d(img2*img2, self.window, padding=self.window_size//2, groups=self.channel) - mu2_sq
        sigma12 = F.conv2d(img1*img2, self.window, padding=self.window_size//2, groups=self.channel) - mu1_mu2
        C1, C2 = 0.01**2, 0.03**2
        ssim_map = ((2*mu1_mu2 + C1)*(2*sigma12 + C2))/((mu1_sq + mu2_sq + C1)*(sigma1_sq + sigma2_sq + C2))
        return ssim_map.mean()

# --- 2. Charbonnier ---
def charbonnier_loss(x, y, eps=1e-3):
    return torch.sqrt((x - y)**2 + eps**2).mean()

# --- 3. Smoothness (Safe & Adaptive) ---
def smoothness_loss_adaptive(pred_disp, img, beta=9.0):
    def gradient_x(x): return F.pad(x, (0, 1, 0, 0))[:, :, :, 1:] - x
    def gradient_y(x): return F.pad(x, (0, 0, 0, 1))[:, :, 1:, :] - x
    
    disp_gradients_x = gradient_x(pred_disp)
    disp_gradients_y = gradient_y(pred_disp)

    image_gradients_x = gradient_x(img)
    image_gradients_y = gradient_y(img)

    weights_x = torch.exp(-torch.mean(torch.abs(image_gradients_x), 1, keepdim=True) * beta)
    weights_y = torch.exp(-torch.mean(torch.abs(image_gradients_y), 1, keepdim=True) * beta)

    smoothness_x = torch.abs(disp_gradients_x) * weights_x
    smoothness_y = torch.abs(disp_gradients_y) * weights_y
    return (smoothness_x + smoothness_y).mean()

# --- HAUPTFUNKTION V9.3 ---
def robust_stereo_loss_v9(outputs, left_img, right_img, gt_disp_L, gt_disp_R=None, 
                           w_geom=0.8, w_photo=1.0, w_lrc=0.5, w_smooth=0.1, w_occ=0.2):
    """
    Version 9.3 (Edge-Aware & Adaptive):
    - Integriert Edge-Weighted Photometric Loss
    - Integriert Adaptive Scale Weights
    - Integriert Dynamic LRC Threshold
    - Integriert Masked Smoothness Loss
    """
    
    # --- Initialisierung ---
    if not hasattr(robust_stereo_loss_v9, 'ssim_module'):
        robust_stereo_loss_v9.ssim_module = SSIM(channel=1).to(left_img.device)
    ssim_loss_fn = robust_stereo_loss_v9.ssim_module
    
    # V9.3 Change: Angepasste Scale Weights
    scale_weights = [1.0, 0.7, 0.3] 
    total_loss, logs = 0.0, {}
    
    _, _, H_full, W_full = left_img.shape

    # --- Helper für Gradienten und Magnitude ---
    def get_gradients(img):
        gx = F.pad(img, (0, 1, 0, 0))[:, :, :, 1:] - img
        gy = F.pad(img, (0, 0, 0, 1))[:, :, 1:, :] - img
        return torch.abs(gx) + torch.abs(gy)

    def get_edge_magnitude(img):
        grads = get_gradients(img)
        return grads.mean(dim=1, keepdim=True)

    # --- Multi-Scale-Loop ---
    for i, weight in enumerate(scale_weights):
        # Aktuelle Prediction holen
        disp_L = outputs["LR"][i]
        disp_R = outputs["RL"][i] if "RL" in outputs else None

        # Aktuelle Auflösung
        _, _, H_curr, W_curr = disp_L.shape

        # --- Scale Adjustments ---
        if W_curr != W_full:
            scale_factor = W_curr / W_full
            # GT skalieren
            gt_L_curr = F.interpolate(gt_disp_L, size=(H_curr, W_curr), mode='nearest-exact') * scale_factor
            # Bilder skalieren
            img_L_curr = F.interpolate(left_img, size=(H_curr, W_curr), mode='bilinear', align_corners=False)
            img_R_curr = F.interpolate(right_img, size=(H_curr, W_curr), mode='bilinear', align_corners=False)
        else:
            gt_L_curr = gt_disp_L
            img_L_curr, img_R_curr = left_img, right_img
            
        # Nan-Schutz
        disp_L = torch.nan_to_num(disp_L, nan=0.0, posinf=192.0, neginf=0.0)
            
        # --- 1. Geometrischer Loss (Supervised) ---
        if gt_L_curr is not None:
            mask_valid = (gt_L_curr > 0) & (gt_L_curr < 192)
            if mask_valid.sum() > 0:
                loss_g = charbonnier_loss(disp_L[mask_valid], gt_L_curr[mask_valid])
                total_loss += w_geom * weight * loss_g
                if i == 0: logs['geom'] = loss_g.item()

        # --- 2. Self-Supervised Losses ---
        if disp_R is not None:
            # Grid
            grid_y_curr, grid_x_curr = torch.meshgrid(torch.arange(H_curr, device=left_img.device), 
                                                      torch.arange(W_curr, device=left_img.device), indexing='ij')
            grid_x_norm = (2.0 * grid_x_curr / (W_curr - 1)) - 1.0
            grid_y_norm = (2.0 * grid_y_curr / (H_curr - 1)) - 1.0
            
            # Warping
            disp_L_norm = 2.0 * disp_L / (W_curr - 1)
            vgrid = torch.stack((grid_x_norm - disp_L_norm.squeeze(1), 
                                 grid_y_norm.expand(disp_L.shape[0], H_curr, W_curr)), dim=3)
            
            disp_R_warped = F.grid_sample(disp_R, vgrid, align_corners=False, padding_mode='border')
            right_warped = F.grid_sample(img_R_curr, vgrid, align_corners=False, padding_mode='border')
            
            # --- LRC Loss ---
            lrc_diff = torch.abs(disp_L - disp_R_warped)
            loss_l = charbonnier_loss(lrc_diff, torch.zeros_like(lrc_diff))
            total_loss += w_lrc * weight * loss_l
            if i == 0: logs['lrc'] = loss_l.item()

            # --- V9.3: Dynamic Threshold & Edge Weights ---
            
            # 1. Dynamic Threshold
            thr_lrc = 1.2 * (W_curr / W_full) # Skaliert mit Auflösung
            mask_vis = (lrc_diff < thr_lrc + 0.05).float().detach()

            # 2. Edge-Aware Weights berechnen
            with torch.no_grad():
                edge_mag = get_edge_magnitude(img_L_curr)
                # Kanten werden doppelt so stark gewichtet wie Flächen
                edge_w = torch.clamp(edge_mag / (edge_mag.mean() + 1e-6), 0.5, 2.0)

            # 3. Edge-Weighted Photometric Loss
            loss_p_charb_edge = (torch.sqrt((img_L_curr - right_warped)**2 + 1e-6) * mask_vis * edge_w).sum() / (mask_vis.sum() + 1e-6)
            
            grad_l = get_gradients(img_L_curr)
            grad_r_w = get_gradients(right_warped)
            # Edge Weight auch auf Gradient anwenden
            loss_p_grad_edge = (torch.abs(grad_l - grad_r_w) * mask_vis * edge_w).sum() / (mask_vis.sum() + 1e-6)

            # SSIM bleibt Standard (da es ein Window-based Loss ist, ist Pixel-Weighting schwierig)
            s_val = ssim_loss_fn(img_L_curr * mask_vis, right_warped * mask_vis)
            loss_p_ssim = 1.0 - s_val
            
            # Neue Gewichtung
            loss_photo = 0.40 * loss_p_ssim + 0.30 * loss_p_charb_edge + 0.30 * loss_p_grad_edge
            
            total_loss += w_photo * weight * loss_photo
            if i == 0: logs['photo'] = loss_photo.item()
            
            # --- V9.3: Masked Smoothness Loss ---
            # Wir wollen Smoothness NUR auf flachen Flächen erzwingen, NICHT an Kanten
            with torch.no_grad():
                # Flat Mask: 1 wo flach, 0 wo Kante
                flat_mask = (edge_mag < edge_mag.mean()).float()
            
            # Hier wenden wir die Flat Mask an
            loss_s = (smoothness_loss_adaptive(disp_L, img_L_curr) * flat_mask).mean()
            total_loss += w_smooth * weight * loss_s # Smoothness jetzt auf allen Scales gewichtet!

    # --- 3. Okklusions-Loss (nur auf höchster Auflösung) ---
    if "RL" in outputs:
        # Full-Res Maske für Occ Head Supervision
        disp_L_full, disp_R_full = outputs["LR"][0], outputs["RL"][0]
        
        # Grid Full
        grid_y, grid_x = torch.meshgrid(torch.arange(H_full, device=left_img.device), 
                                        torch.arange(W_full, device=left_img.device), indexing='ij')
        grid_x_norm = (2.0 * grid_x / (W_full - 1)) - 1.0
        grid_y_norm = (2.0 * grid_y / (H_full - 1)) - 1.0
        
        vgrid_full = torch.stack((grid_x_norm - (2.0*disp_L_full/(W_full-1)).squeeze(1), 
                                  grid_y_norm.expand(disp_L_full.shape[0],H_full,W_full)), dim=3)
        disp_R_warped_full = F.grid_sample(disp_R_full, vgrid_full, align_corners=False, padding_mode='border')
        lrc_diff_full = torch.abs(disp_L_full - disp_R_warped_full)
        mask_vis_full = (lrc_diff_full < 1.2).float().detach()

        # Supervision für den Occ Head
        if len(outputs["LR"]) > 3: # Sicherstellen, dass occ_logits existieren
            loss_o = F.binary_cross_entropy_with_logits(outputs["LR"][3], 1.0 - mask_vis_full)
            total_loss += w_occ * loss_o
            logs['occ'] = loss_o.item()
    
    return total_loss, logs

In [8]:
# --- 1. Externer Warper (NUR für Training & Loss) ---
class TrainingWarper(nn.Module):
    def __init__(self):
        super().__init__()
        self.grid_cache = {}
    
    def forward(self, img, disp):
        B, _, H, W = img.shape
        device = img.device
        key = (H, W, device)
        
        if key not in self.grid_cache:
            y, x = torch.meshgrid(torch.arange(H, device=device), torch.arange(W, device=device), indexing='ij')
            self.grid_cache[key] = ( (2.0*x/(W-1))-1.0, (2.0*y/(H-1))-1.0 )
            
        grid_x, grid_y = self.grid_cache[key]
        
        # Warping Vektor: Pixel (u, v) -> (u - disp, v)
        disp_norm = 2.0 * disp / (W - 1)
        vgrid = torch.stack([grid_x - disp_norm.squeeze(1), grid_y.expand(B,H,W)], dim=3)
        
        return F.grid_sample(img, vgrid, align_corners=False, padding_mode='border')

# --- 2. Deine Debug- & Save-Funktionen ---

def get_full_res_preds(model, dataset, index, device='cuda'):
    """Interne Hilfsfunktion: Holt Modell-Output und skaliert ALLES auf 640x480."""
    model.eval()
    with torch.no_grad():
        l, r, dl, dr = dataset[index]
        l_in = l.unsqueeze(0).to(device)
        r_in = r.unsqueeze(0).to(device)
        
        # 1. Forward Pass
        disp_final, disp_v1, disp_low, occ_logits = model.core(l_in, r_in)
        
        # 2. Skalierungs-Logik (bringt alles auf 640x480)
        def up(t):
            if t is None: return None
            curr_w = t.shape[-1]
            scale = 640 / curr_w
            return F.interpolate(t, size=(480, 640), mode='bilinear', align_corners=False) * scale

        # Occ-Logits brauchen keine wertmäßige Skalierung, nur Auflösung
        occ_prob = torch.sigmoid(F.interpolate(occ_logits, size=(480, 640), mode='bilinear', align_corners=False))
        
        # 3. RL Pass für Symmetrie/LRC
        l_f, r_f = torch.flip(l_in, [3]), torch.flip(r_in, [3])
        disp_RL_f, _, _, _ = model.core(r_f, l_f)
        disp_RL = torch.flip(up(disp_RL_f), [3])
        
        # 4. Echo-Map (LRC) Berechnung auf Full Res
        disp_LR = up(disp_final)
        B, _, H, W = disp_LR.shape
        grid_x = torch.arange(W, device=device).view(1, 1, 1, W).expand(B, 1, H, W).float()
        grid_y = torch.arange(H, device=device).view(1, 1, H, 1).expand(B, 1, H, W).float()
        x_proj = grid_x - disp_LR
        norm_x = 2.0 * x_proj / (W - 1) - 1.0
        norm_y = 2.0 * grid_y / (H - 1) - 1.0
        grid = torch.stack((norm_x.squeeze(1), norm_y.squeeze(1)), dim=3)
        disp_RL_warped = F.grid_sample(disp_RL, grid, align_corners=False, padding_mode='border')
        echo_map = torch.abs(disp_LR - disp_RL_warped)

        # Alles nach Numpy [H, W]
        def to_np(t): return t.squeeze().cpu().numpy()
        
        return {
            "l": to_np(l), "dl": to_np(dl), 
            "final": to_np(disp_LR), "v1": to_np(up(disp_v1)), "low": to_np(up(disp_low)),
            "occ": to_np(occ_prob), "echo": to_np(echo_map), "rl_warp": to_np(disp_RL_warped)
        }

def save_debug_visuals(model, dataset, epoch, index=6, device='cuda'):
    d = get_full_res_preds(model, dataset, index, device)
    fig, axs = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f"Debug Visuals - Epoche {epoch}", fontsize=16)

    axs[0,0].imshow(d['l'], cmap='gray'); axs[0,0].set_title("Input Left")
    axs[0,1].imshow(d['dl'], cmap='magma', vmin=0, vmax=192); axs[0,1].set_title("Ground Truth")
    
    im_f = axs[0,2].imshow(d['final'], cmap='magma', vmin=0, vmax=192)
    axs[0,2].set_title("Final Prediction (1/1)"); fig.colorbar(im_f, ax=axs[0,2])

    axs[1,0].imshow(d['low'], cmap='magma', vmin=0, vmax=192); axs[1,0].set_title("Stage Low (1/4)")
    axs[1,1].imshow(d['v1'], cmap='magma', vmin=0, vmax=192); axs[1,1].set_title("Stage V1 (1/2)")
    
    # EPE Map
    epe = np.abs(d['dl'] - d['final'])
    epe[(d['dl'] <= 0) | (d['dl'] >= 192)] = 0
    im_e = axs[1,2].imshow(epe, cmap='jet', vmin=0, vmax=10)
    axs[1,2].set_title("EPE Map (Error)"); fig.colorbar(im_e, ax=axs[1,2])

    for ax in axs.flatten(): ax.axis('off')
    plt.tight_layout(); os.makedirs("training_progress", exist_ok=True)
    plt.savefig(f"training_progress/FusedBackbone-Stereo_debug_ep_{epoch:03d}.png"); plt.close()

def save_preview(model, dataset, name, index=6, device='cuda'):
    d = get_full_res_preds(model, dataset, index, device)
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f"Preview Analysis: {name}", fontsize=16)

    axes[0,0].imshow(d['l'], cmap='gray'); axes[0,0].set_title("Input")
    axes[0,1].imshow(d['dl'], cmap='magma', vmin=0, vmax=192); axes[0,1].set_title("GT")
    im_p = axes[0,2].imshow(d['final'], cmap='magma', vmin=0, vmax=192)
    axes[0,2].set_title("Prediction"); fig.colorbar(im_p, ax=axes[0,2])

    im_ec = axes[1,0].imshow(d['echo'], cmap='hot', vmin=0, vmax=5)
    axes[1,0].set_title("Echo-Map (LRC Error)"); fig.colorbar(im_ec, ax=axes[1,0])

    im_oc = axes[1,1].imshow(d['occ'], cmap='gray', vmin=0, vmax=1)
    axes[1,1].set_title("Predicted Occlusion"); fig.colorbar(im_oc, ax=axes[1,1])

    epe = np.abs(d['dl'] - d['final'])
    epe[(d['dl'] <= 0) | (d['dl'] >= 192)] = 0
    im_ep = axes[1,2].imshow(epe, cmap='jet', vmin=0, vmax=10)
    axes[1,2].set_title("EPE Error Map"); fig.colorbar(im_ep, ax=axes[1,2])

    for ax in axes.flatten(): ax.axis('off')
    plt.tight_layout(); os.makedirs("training_progress", exist_ok=True)
    plt.savefig(f"training_progress/FusedBackbone-Stereo_preview_{name}.png"); plt.close()

def plot_disparity_profile(model, dataset, epoch, index=0, device='cuda'):
    d = get_full_res_preds(model, dataset, index, device)
    H, W = d['final'].shape
    rows = [int(H * 0.25), int(H * 0.50), int(H * 0.75)]
    labels = ["25%", "50%", "75%"]
    
    fig, axs = plt.subplots(4, 1, figsize=(12, 16))
    axs[0].imshow(d['l'], cmap='gray')
    for r in rows: axs[0].axhline(r, color='yellow', linewidth=2)
    axs[0].set_title(f"Profile Lines (Epoch {epoch})"); axs[0].axis('off')

    for i, (r, lbl) in enumerate(zip(rows, labels)):
        ax = axs[i+1]
        ax.plot(d['dl'][r, :], 'k-', label='Ground Truth', linewidth=2)
        ax.plot(d['final'][r, :], 'r-', label='Final Prediction', alpha=0.8)
        ax.plot(d['v1'][r, :], 'g--', label='V1 Coarse', alpha=0.6)
        ax.set_title(f"Profile at {lbl} Height (y={r})")
        ax.set_ylim(-5, 200); ax.grid(True, alpha=0.3)
        if i==0: ax.legend()

    plt.tight_layout(); os.makedirs("training_progress", exist_ok=True)
    plt.savefig(f"training_progress/FusedBackbone-Stereo_profile_ep{epoch:03d}.png"); plt.close()

def save_symmetry_comparison(model, dataset, device, epoch=0, index=0):
    d = get_full_res_preds(model, dataset, index, device)
    fig, axs = plt.subplots(2, 2, figsize=(12, 8))
    
    im1 = axs[0,0].imshow(d['final'], cmap='magma', vmin=0, vmax=192)
    axs[0,0].set_title("LR Prediction"); fig.colorbar(im1, ax=axs[0,0])
    
    im2 = axs[0,1].imshow(d['rl_warp'], cmap='magma', vmin=0, vmax=192)
    axs[0,1].set_title("RL Prediction (Warped to Left)")
    
    im3 = axs[1,0].imshow(d['echo'], cmap='hot', vmin=0, vmax=10)
    axs[1,0].set_title("LRC Difference"); fig.colorbar(im3, ax=axs[1,0])
    
    axs[1,1].imshow(d['l'], cmap='gray'); axs[1,1].set_title("Left Image Input")

    for ax in axs.flat: ax.axis('off')
    plt.tight_layout(); os.makedirs("training_progress", exist_ok=True)
    plt.savefig(f"training_progress/FusedBackbone-Stereo_symmetry_ep{epoch:03d}.png"); plt.close()

# --- 3. Trainings-Loop (V9.5) ---

def train_strategic_v9(
    model, 
    base_dir,
    start_epoch=0, 
    resume_checkpoint_path=None,
    # --- Training Hyperparameter ---
    epochs=75, 
    lr_max=2e-4, 
    weight_decay=1e-5,
    warmup_pct=0.1,      
    grad_clip=1.0,
    accumulation_steps=12, 
    
    # --- Dataloader & Dataset ---
    batch_size=4, 
    num_workers=4, 
    use_crop=True,        
    use_aug=True
):
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"🚀 V9 TRAINING START | Device: {device} | BS={batch_size} | Eff. BS={batch_size * accumulation_steps}")
    
    # 1. Datasets & Loader
    train_ds = StereoDataset(base_dir, mode='train', use_crop=use_crop, use_augmentation=use_aug)
    val_ds = StereoDataset(base_dir, mode='val', use_crop=False, use_augmentation=False)
    
    train_loader = DataLoader(
        train_ds, 
        batch_size=batch_size, 
        shuffle=True, 
        num_workers=num_workers, 
        pin_memory=True, 
        persistent_workers=True,
        prefetch_factor=2,
        drop_last=True
    )
    
    val_loader = DataLoader(
        val_ds, 
        batch_size=1, 
        shuffle=False, 
        num_workers=2,
        pin_memory=True
    )

    optimizer = optim.AdamW(model.parameters(), lr=lr_max, weight_decay=weight_decay)
    
    effective_steps_per_epoch = len(train_loader) // accumulation_steps
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr_max, total_steps=epochs * effective_steps_per_epoch,
        pct_start=warmup_pct, div_factor=25, final_div_factor=1000
    )

    # --- Optimizer & Scheduler laden ---
    if resume_checkpoint_path is not None:
        print("   🔧 Lade Optimizer & Scheduler State...")
        checkpoint = torch.load(resume_checkpoint_path, map_location=device)
        try:
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            # Scheduler laden ist tricky bei OneCycleLR, oft ist ein Neustart besser.
            # Aber wenn du genau weitermachen willst:
            # scheduler.load_state_dict(checkpoint['scheduler_state_dict']) 
            print("   ✅ Optimizer geladen.")
        except Exception as e:
            print(f"   ⚠️ Konnte Optimizer nicht laden: {e}")
    
    # FP16 Scaler
    scaler = GradScaler()
    warper = TrainingWarper().to(device) # Externer Warper für Training
    
    current_best_epe = float('inf')
    
    log_file = "training_log_FusedBackbone-Stereo.txt"
    with open(log_file, "w") as f:
        f.write("epoch\tavg_loss\tval_epe\tgeom\tphoto\tlrc\tocc\tlr\n")

    optimizer.zero_grad()

    for epoch in range(start_epoch, epochs):
        model.train()
        sum_logs = {"geom": 0.0, "photo": 0.0, "lrc": 0.0, "occ": 0.0}
        epoch_loss = 0.0
        grad_norm = 0.0
        # --- PHASE 4: SNIPER MODE (Epoche 75+) ---
        if epoch >= 75:
            phase_name = "SNIPER"
            
            # 1. Loss Gewichte (Präzision > Glättung)
            w_geom   = 1.2   # Der Chef (Ground Truth)
            w_photo  = 0.4   # Kleiner Realitäts-Check
            w_lrc    = 0.3   # Sicherheitsnetz
            w_smooth = 0.01  # MINIMAL! Wichtig für Schärfe (nicht 0.03!)
            w_occ    = 0.3   # Stabilisator
            
            # 2. Clipping (Enger Gürtel)
            current_clip = 1.0 
            
            # 3. Manuelle Lernrate (Scheduler ignorieren/überschreiben)
            # Wir wollen konstant niedrig fliegen.
            current_lr = 3e-5 
            for param_group in optimizer.param_groups:
                param_group['lr'] = current_lr
                
            # WICHTIG: Backbone muss im Eval-Modus bleiben, auch wenn model.train() aufgerufen wurde
            model.backbone.eval()
        # --- Curriculum Learning Strategy ---
        # Epoche 0-4: Geometrie-Drill (Harter Lehrer)
        elif epoch < 5:
            w_geom, w_photo, w_smooth, w_lrc, w_occ = 2.0, 0.0, 0.0, 0.1, 0.1
            current_clip = 1.0 
            phase_name = "DRILL"
            
        # Epoche 5-29: Extended Fade-In & Stability (Der "Sweet Spot")
        # Wir strecken die Phase, die so gut funktionierte, bis Epoche 30.
        elif epoch < 30:
            # Wir interpolieren viel langsamer. 
            # Ziel bei Epoche 30: Geom 1.0, Photo 0.8
            # p läuft von 0.0 bis 1.0 über den Zeitraum Epoche 5 bis 30
            p = (epoch - 5) / 25.0 
            
            w_geom = 2.0 - (1.0 * p)   # Sinkt sanft von 2.0 auf 1.0
            w_photo = 0.8 * p          # Steigt sanft auf 0.8
            w_smooth = 0.03 * p        # Bleibt sehr niedrig (max 0.03!)
            w_lrc = 0.1 + (0.4 * p)    # Steigt moderat auf 0.5
            w_occ = 0.2
            
            current_clip = 2.0
            phase_name = "STABILITY"

        # Epoche 30+: High-End Refinement (statt "Matsch-Full")
        else:
            # Hier greift jetzt das korrigierte Setting:
            # Geom bleibt stark, Smoothness wird fast abgeschaltet.
            w_geom = 1.0    # Bleibt der Anker
            w_photo = 1.0   # Jetzt darf Photo voll mitreden
            w_smooth = 0.01 # Minimal! Nur gegen Rauschen, nicht gegen Kanten.
            w_lrc = 0.5     # Nicht höher gehen, sonst Ghosting
            w_occ = 0.3
            
            current_clip = 5.0
            phase_name = "REFINEMENT"
        
        pbar = tqdm(train_loader, desc=f"Ep {epoch+1} [{phase_name}]", ncols=160)
        
        for batch_idx, (l, r, dl, dr) in enumerate(pbar):
            l, r = l.to(device), r.to(device)
            dl, dr = dl.to(device), dr.to(device)
            curr_lr = scheduler.get_last_lr()[0]
            
            with autocast(): # FP16 Context
                # --- 1. Forward Pass LR ---
                out_LR = model.core(l, r) 
                
                # --- 2. Forward Pass RL (Siamese) ---
                l_flip, r_flip = torch.flip(l, [3]), torch.flip(r, [3])
                out_RL_raw = model.core(r_flip, l_flip)
                out_RL = tuple(torch.flip(o, [3]) for o in out_RL_raw)
                
                # --- 3. Occ Supervision Logic ---
                # Berechne geometrische Okklusions-Maske hier im Loop
                disp_LR_final = out_LR[0]
                disp_RL_final = out_RL[0]
                disp_RL_warped = warper(disp_RL_final, disp_LR_final)
                lrc_diff = torch.abs(disp_LR_final - disp_RL_warped)
                gt_occ_mask = (lrc_diff > 1.5).float().detach()

                outputs = {"LR": out_LR, "RL": out_RL}
                
                # --- 4. Calculate Loss ---
                loss, logs = robust_stereo_loss_v9(
                    outputs, l, r, dl, gt_disp_R=dr,
                    w_geom=w_geom, w_photo=w_photo, w_lrc=w_lrc, 
                    w_smooth=w_smooth, w_occ=w_occ
                )
                
                # Occ Head Loss dazu
                if out_LR[3] is not None:
                    loss_occ = F.binary_cross_entropy_with_logits(out_LR[3], gt_occ_mask)
                    loss += w_occ * loss_occ # Gewichtung für Occ Head

                # Loss Skalierung für Akkumulation
                loss_scaled = loss / accumulation_steps
            
            # Backward mit Scaler
            scaler.scale(loss_scaled).backward()
            
            # Logging
            epoch_loss += loss.item()
            for k, v in logs.items(): sum_logs[k] += v

            # Optimization Step
            if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), current_clip)
                
                scale_before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                scale_after = scaler.get_scale()
                
                if scale_after >= scale_before:
                    scheduler.step()
                
                optimizer.zero_grad() 
            
            vram_gb = torch.cuda.memory_reserved(device) / 1024**3
            pbar.set_postfix({
                'VRAM': f"{vram_gb:.1f}G",
                'Loss': f"{loss.item():.2f}",
                'Geom': f"{logs.get('geom', 0):.2f}",
                'Photo': f"{logs.get('photo', 0):.2f}",
                'LRC': f"{logs.get('lrc', 0):.2f}",
                'OCC': f"{logs.get('occ', 0):.2f}",
                'Clip': f"{current_clip:.1f}"
            })

        # Validation & Logging
        # (Hinweis: validate Funktion muss existieren oder hier eingefügt werden)
        # avg_loss = epoch_loss / len(train_loader)
        # val_loss, val_epe = validate(model, val_loader, device) 
        
        # Placeholder Validierung, falls validate() nicht definiert ist:
        avg_loss = epoch_loss / len(train_loader)
        val_loss, val_epe = validate(model, val_loader, device)
        
        avg_logs = {k: v / len(train_loader) for k, v in sum_logs.items()}
        
        print(f"📈 Ep {epoch+1}: Loss: {avg_loss:.4f} | EPE: {val_epe:.2f}")
        
        final_lr = scheduler.get_last_lr()[0]
        with open(log_file, "a") as f:
            f.write(f"{epoch+1}\t{avg_loss:.4f}\t{val_epe:.4f}\t"
                    f"{avg_logs.get('geom', 0):.4f}\t"
                    f"{avg_logs.get('photo', 0):.4f}\t"
                    f"{avg_logs.get('lrc', 0):.4f}\t"
                    f"{avg_logs.get('occ', 0):.4f}\t"
                    f"{final_lr:.2e}\n")
        # Visuals
        save_debug_visuals(model, val_loader.dataset, epoch+1, index=6)
        plot_disparity_profile(model, val_loader.dataset, epoch+1, index=6)
        save_preview(model, val_loader.dataset, f"epoch_{(epoch+1):03d}", index=6)
        save_symmetry_comparison(model, val_loader.dataset, device, epoch=epoch+1, index=6)

        # Save Checkpoints
        if val_epe < current_best_epe:
            current_best_epe = val_epe
            torch.save(model.state_dict(), "FusedBackbone-Stereo_BEST.pth")
        if (epoch + 1) % 5 == 0:
            torch.save(model.state_dict(), f"FusedBackbone-Stereo_ep_{epoch+1}.pth")
            
    print("✅ V9 Training Complete.")
     




In [11]:
# --- CELL 4: MAIN V9 (MIT RESTART-LOGIK) ---

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def init_weights_v9(m):
    """
    Spezielle Initialisierung für V9:
    Wir nutzen Kaiming Init für unsere neuen Layer (Refiner, CostVol, Heads), 
    aber lassen den Pre-Trained Backbone in Ruhe!
    """
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        # ReLU-optimiertes Init (Kaiming / He)
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)
    elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
        nn.init.constant_(m.weight, 1)
        nn.init.constant_(m.bias, 0)

def main():
    # 1. Reproduzierbarkeit & Cleanup
    set_seed(42)
    gc.collect()
    torch.cuda.empty_cache()
    
    # 2. Modell V9 instanziieren
    print("🏗️ Erstelle StereoNet V9 (MobileNetV3-Large + Mini U-Nets)...")
    model = StereoNet_NPU_V9(max_disp=192).to(device)
    
    # --- RESTART LOGIK START (ROBUST) ---
    checkpoint_path = "FusedBackbone-Stereo_BEST.pth"  
    start_epoch = 14 # Wir erzwingen Start bei 15, da Checkpoint wahrscheinlich von 14 ist.
    
    if os.path.exists(checkpoint_path):
        print(f"🔄 Checkpoint gefunden: {checkpoint_path}")
        print("   Lade Gewichte...")
        
        checkpoint = torch.load(checkpoint_path, map_location=device)
        
        # Check: Ist es ein Dict mit Metadaten oder nur das State Dict?
        if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
            # Fall A: Voller Checkpoint (mit Optimizer etc.)
            print("   -> Format: Full Checkpoint (Dict)")
            model.load_state_dict(checkpoint['model_state_dict'])
        else:
            # Fall B: Nur Gewichte (Direct State Dict)
            print("   -> Format: Weights Only (Direct State Dict)")
            model.load_state_dict(checkpoint)
            
        print(f"   🚀 Setze Training fort ab Epoche {start_epoch} (Phase: STABILITY)")
        
    else:
        print("🆕 Kein Checkpoint gefunden. Starte Training von Null.")
    # --- RESTART LOGIK ENDE ---

    # Anpassung für kleine Batchsizes (BS=4):
    model.apply(lambda m: setattr(m, 'momentum', 0.01) if isinstance(m, nn.BatchNorm2d) else None)
    
    # 4. Pfad Konfiguration
    dataset_path = r"/home/slarc/datasets/sceneflow" 
    
    if not os.path.exists(dataset_path):
        print(f"⚠️ KRITISCH: Pfad {dataset_path} nicht gefunden!")
        return
    
    # 5. Training Starten
    # WICHTIG: train_strategic_v9 muss 'start_epoch' und 'resume_checkpoint' unterstützen!
    # Ich übergebe hier den Pfad, damit die Funktion den Optimizer laden kann.
    train_strategic_v9(
        model=model,
        base_dir=dataset_path,
        
        # Restart Parameter
        start_epoch=start_epoch,           # Neu: Damit der Loop nicht bei 0 anfängt
        resume_checkpoint_path=checkpoint_path if os.path.exists(checkpoint_path) else None, # Neu
        
        # Hyperparameter für V9
        epochs=75,          
        lr_max=2e-4,        
        weight_decay=1e-5,  
        warmup_pct=0.1,      
        grad_clip=1.0,
        
        # Hardware & Data
        batch_size=4,        # Optimiert
        accumulation_steps=12,
        num_workers=4,
        use_crop=True,       
        use_aug=False        
        )

if __name__ == '__main__':
    main()

    


🏗️ Erstelle StereoNet V9 (MobileNetV3-Large + Mini U-Nets)...


Unexpected keys (classifier.bias, classifier.weight, conv_head.bias, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.
/tmp/ipykernel_348677/3704730142.py:42: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file

🔄 Checkpoint gefunden: FusedBackbone-Stereo_BEST.pth
   Lade Gewichte...
   -> Format: Weights Only (Direct State Dict)
   🚀 Setze Training fort ab Epoche 14 (Phase: STABILITY)
🚀 V9 TRAINING START | Device: cuda | BS=4 | Eff. BS=48
[TRAIN] Scanne Bilder in: /home/slarc/datasets/sceneflow/FlyingThings3D_subset_image_clean/FlyingThings3D_subset/train/image_clean
[TRAIN] 21818 Paare gefunden.
[VAL] Scanne Bilder in: /home/slarc/datasets/sceneflow/FlyingThings3D_subset_image_clean/FlyingThings3D_subset/val/image_clean
[VAL] 4248 Paare gefunden.
   🔧 Lade Optimizer & Scheduler State...


/tmp/ipykernel_348677/1618157576.py:223: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(resume_checkpoint_path, map_location=device)
/tmp/ipykernel_34

   ⚠️ Konnte Optimizer nicht laden: 'optimizer_state_dict'


Ep 15 [STABILITY]:   0%|                                                                                                               | 0/5454 [00:00<?, ?it/s]/tmp/ipykernel_348677/1618157576.py:316: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(): # FP16 Context
Ep 15 [STABILITY]:   0%|                           | 11/5454 [00:06<55:41,  1.63it/s, VRAM=7.7G, Loss=8.56, Geom=2.76, Photo=0.03, LRC=4.66, OCC=0.26, Clip=2.0]


KeyboardInterrupt: 

In [10]:
# --- CELL 4b: SNIPER MODE RESTART (EPOCHE 75+) ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def init_weights_v9(m):
    """
    Spezielle Initialisierung für V9:
    Wir nutzen Kaiming Init für unsere neuen Layer (Refiner, CostVol, Heads), 
    aber lassen den Pre-Trained Backbone in Ruhe!
    """
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        # ReLU-optimiertes Init (Kaiming / He)
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)
    elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
        nn.init.constant_(m.weight, 1)
        nn.init.constant_(m.bias, 0)
def main_sniper():
    # 1. Reproduzierbarkeit & Cleanup
    set_seed(42)
    gc.collect()
    torch.cuda.empty_cache()
    
    # 2. Modell V9 instanziieren
    print("🏗️ Erstelle StereoNet V9 für Sniper Mode...")
    model = StereoNet_NPU_V9(max_disp=192).to(device)
    
    # --- RESTART LOGIK (SNIPER MODE) ---
    # WICHTIG: Hier den Pfad zum letzten Checkpoint (Epoche 74/75) angeben!
    checkpoint_path = "FusedBackbone-Stereo_ep_75.pth"   
    start_epoch = 75  # Wir starten explizit im Sniper Mode
    
    if os.path.exists(checkpoint_path):
        print(f"🔄 Checkpoint gefunden: {checkpoint_path}")
        print("   Lade Gewichte...")
        
        checkpoint = torch.load(checkpoint_path, map_location=device)
        
        # Robustes Laden
        if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
            print("   -> Format: Full Checkpoint (Dict)")
            model.load_state_dict(checkpoint['model_state_dict'])
        else:
            print("   -> Format: Weights Only (Direct State Dict)")
            model.load_state_dict(checkpoint)
            
        print(f"   🚀 Setze Training fort ab Epoche {start_epoch} (Phase: SNIPER MODE)")
        
    else:
        print(f"⚠️ KRITISCH: Kein Checkpoint unter {checkpoint_path} gefunden!")
        print("   Für Sniper Mode benötigen wir ein vortrainiertes Modell!")
        return 

    # --- SNIPER MODE SPEZIAL: FREEZE BACKBONE ---
    print("❄️ FREEZING BACKBONE & Fixing BatchNorm stats...")
    
    # 1. Gradients ausschalten (Spart VRAM & schützt Features)
    for param in model.backbone.parameters():
        param.requires_grad = False
    
    # 2. Backbone in Eval-Modus zwingen (Verhindert BN-Update bei kleiner Batchsize)
    model.backbone.eval()
    # --------------------------------------------

    # Anpassung für kleine Batchsizes bei den noch trainierbaren Teilen
    # (Nur für BN-Layer, die NICHT im Backbone sind)
    for name, m in model.named_modules():
        if isinstance(m, nn.BatchNorm2d) and "backbone" not in name:
            m.momentum = 0.01
    
    # 4. Pfad Konfiguration
    dataset_path = r"/home/slarc/datasets/sceneflow" 
    
    if not os.path.exists(dataset_path):
        print(f"⚠️ KRITISCH: Pfad {dataset_path} nicht gefunden!")
        return
    
    # 5. Training Starten (Sniper Config)
    train_strategic_v9(
        model=model,
        base_dir=dataset_path,
        
        # Restart Parameter
        start_epoch=start_epoch, 
        resume_checkpoint_path=checkpoint_path, 
        
        # Hyperparameter für SNIPER PHASE
        epochs=100,         # Verlängert auf 100
        lr_max=3e-5,        # Sehr niedrige Start-LR (wird im Loop eh überschrieben)
        weight_decay=1e-5,  
        warmup_pct=0.0,     # Kein Warmup nötig
        grad_clip=1.0,      # ENGER Clip für Stabilität
        
        # Hardware & Data
        batch_size=4,       
        accumulation_steps=12,
        num_workers=4,
        use_crop=True,       
        use_aug=False       # Augmentation aus für reine Geometrie-Präzision
        )

if __name__ == '__main__':
    main_sniper()

🏗️ Erstelle StereoNet V9 für Sniper Mode...


Unexpected keys (classifier.bias, classifier.weight, conv_head.bias, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


🔄 Checkpoint gefunden: FusedBackbone-Stereo_ep_75.pth
   Lade Gewichte...
   -> Format: Weights Only (Direct State Dict)
   🚀 Setze Training fort ab Epoche 75 (Phase: SNIPER MODE)
❄️ FREEZING BACKBONE & Fixing BatchNorm stats...
🚀 V9 TRAINING START | Device: cuda | BS=4 | Eff. BS=48
[TRAIN] Scanne Bilder in: /home/slarc/datasets/sceneflow/FlyingThings3D_subset_image_clean/FlyingThings3D_subset/train/image_clean


/tmp/ipykernel_349326/3266463301.py:41: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location=device)
/tmp/ipykernel_349326/161

[TRAIN] 21818 Paare gefunden.
[VAL] Scanne Bilder in: /home/slarc/datasets/sceneflow/FlyingThings3D_subset_image_clean/FlyingThings3D_subset/val/image_clean
[VAL] 4248 Paare gefunden.
   🔧 Lade Optimizer & Scheduler State...
   ⚠️ Konnte Optimizer nicht laden: 'optimizer_state_dict'


...

Ep 88 [SNIPER]: 100%|████████████████████████████| 5454/5454 [21:59<00:00,  4.13it/s, VRAM=6.6G, Loss=5.14, Geom=1.90, Photo=0.03, LRC=3.32, OCC=0.34, Clip=1.0]
                                                                                                                                                      

📈 Ep 88: Loss: 5.0134 | EPE: 1.66


Ep 89 [SNIPER]: 100%|████████████████████████████| 5454/5454 [21:30<00:00,  4.23it/s, VRAM=6.6G, Loss=5.03, Geom=1.38, Photo=0.02, LRC=5.11, OCC=0.27, Clip=1.0]
                                                                                                                                                      

📈 Ep 89: Loss: 4.9921 | EPE: 1.69


Ep 90 [SNIPER]:  81%|██████████████████████▋     | 4413/5454 [17:48<04:02,  4.30it/s, VRAM=6.6G, Loss=4.29, Geom=1.21, Photo=0.01, LRC=4.06, OCC=0.31, Clip=1.0]IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

Ep 96 [SNIPER]:  76%|█████████████████████▍      | 4169/5454 [16:52<04:55,  4.36it/s, VRAM=6.6G, Loss=3.27, Geom=0.94, Photo=0.02, LRC=3.01, OCC=0.23, Clip=1.0]IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [10]:
%abort

UsageError: Line magic function `%abort` not found.


In [ ]:
# --- DEBUG CELL: INSPECT DATA VALUES ---
import matplotlib.pyplot as plt

# Wir nutzen deine Dataset-Instanzlogik, um einen Pfad zu finden
ds_debug = StereoDataset(r"/home/slarc/datasets/sceneflow", mode='train', use_crop=False)
idx = 0 
l_path = ds_debug.left_files[idx]
disp_path = ds_debug.disp_left_files[idx]

print(f"🔍 Untersuche Datei: {disp_path}")

# Manueller Load (Kopie deiner Logik)
def debug_load_pfm(file):
    with open(file, "rb") as f:
        header = f.readline().rstrip()
        dim_match = re.match(rb'^(\d+)\s(\d+)\s$', f.readline())
        width, height = map(int, dim_match.groups())
        scale = float(f.readline().rstrip())
        endian = '<' if scale < 0 else '>'
        data = np.fromfile(f, endian + 'f')
        data = np.flipud(data.reshape(height, width))
    return data, scale, width, height

raw_disp, raw_scale, w, h = debug_load_pfm(disp_path)

print(f"--- RAW PFM STATS ---")
print(f"Original Size: {w}x{h}")
print(f"Scale Factor in Header: {raw_scale}")
print(f"Min Wert: {raw_disp.min():.4f}")
print(f"Max Wert: {raw_disp.max():.4f}")
print(f"Mean Wert: {raw_disp.mean():.4f}")
print(f"NaNs: {np.isnan(raw_disp).sum()}")
print(f"Infs: {np.isinf(raw_disp).sum()}")

# Validitäts-Check
valid_mask = (raw_disp > 0) & (raw_disp < 192)
print(f"Pixel im Bereich 0-192 (Valid): {valid_mask.sum()} von {raw_disp.size} ({valid_mask.sum()/raw_disp.size:.2%})")

# Resize Check
target_w = 640
scale_x = target_w / w
resized_disp = cv2.resize(raw_disp, (640, 480), interpolation=cv2.INTER_LINEAR) * scale_x

print(f"\n--- RESIZED STATS (Target) ---")
print(f"Resized Min: {resized_disp.min():.4f}")
print(f"Resized Max: {resized_disp.max():.4f}")
print(f"Valid Pixels nach Resize: {(resized_disp > 0).sum()}")

# Histogramm
plt.figure(figsize=(10,4))
plt.hist(raw_disp.flatten(), bins=100, range=(0, 300))
plt.title("Disparitäts-Verteilung (Raw)")
plt.xlabel("Disparität (px)")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
full_dataset = StereoDataset(
        left_dir='/home/slarc/datasets/sceneflow/left',
        right_dir='/home/slarc/datasets/sceneflow/right',
        disp_dir='/home/slarc/datasets/sceneflow/disp',
        training=True
    )
val_ratio = 0.1
val_size = int(len(full_dataset) * val_ratio)
train_size = len(full_dataset) - val_size

train_dataset, val_dataset = torch.utils.data.random_split(
        full_dataset,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )

# Ein Bild aus dem Dataset holen
left, right, gt = train_dataset[7491] 

plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.imshow(left.squeeze(), cmap='gray')
plt.title("Eingangsbild (Ist es aufrecht?)")

plt.subplot(1, 3, 2)
plt.imshow(gt.squeeze(), cmap='jet')
plt.title("GT Disparität")

# Check: Wo sind die Gradienten am stärksten?
# Wenn dy > dx, dann ist die Disparität vertikal orientiert!
dy, dx = torch.gradient(gt.squeeze())
plt.subplot(1, 3, 3)
plt.imshow(dx.abs() > dy.abs(), cmap='gray')
plt.title("Weiß = Horizontale Struktur\nSchwarz = Vertikale Struktur")
plt.show()

In [ ]:
# Testen Sie:
feat = make_feature_extractor()
dummy = torch.randn(1, 1, 480, 640)
out = feat(dummy)
print(f"Feature channels: {out.shape[1]}")  # Muss 32 sein!

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from torchvision import transforms
from PIL import Image

# ---------------------------------------------------------
# Hilfsfunktion: Bild laden und normalisieren
# ---------------------------------------------------------
def load_gray_image(path):
    img = Image.open(path).convert("L")
    t = transforms.ToTensor()
    return t(img).unsqueeze(0).cuda()

# Pfade und Device
left_path  = "/home/slarc/datasets/sceneflow/left/0000006.png"
right_path = "/home/slarc/datasets/sceneflow/right/0000006.png"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

left  = load_gray_image(left_path)
right = load_gray_image(right_path)

# Modell laden (Stelle sicher, dass die Klasse definiert ist)
model = StereoFusionAllMode().to(device)
model.eval()

with torch.no_grad():
    # 1. Fast Mode (Nur LR Pass)
    out_fast = model(left, right, mode="fast")
    # Extraktion aus dem Dictionary-Key "LR"
    disp_fast = out_fast["LR"][0]
    occ_fast  = out_fast["LR"][3]

    # 2. Precise Mode (LR und geflippter RL Pass)
    out_prec = model(left, right, mode="precise")
    disp_prec_LR = out_prec["LR"][0]   # Normaler Pass
    disp_prec_RL = out_prec["RL"][0]   # Symmetrischer RL-Pass (bereits zurückgeflippt!)
    
    # Echo-Analyse: Wo unterscheiden sich LR und RL? (Meist am linken Rand)
    echo_map = torch.abs(disp_prec_LR - disp_prec_RL)

# ---------------------------------------------------------
# Visualisierung: Der "Miststück-Check"
# ---------------------------------------------------------
def show_disp(disp, title, subplot_pos, cmap="magma"):
    plt.subplot(2, 2, subplot_pos)
    disp_np = disp.squeeze().cpu().numpy()
    plt.imshow(disp_np, cmap=cmap, vmin=0, vmax=192)
    plt.colorbar(label="Pixel")
    plt.title(title)
    plt.axis("off")

plt.figure(figsize=(16, 10))

# Oben Links: Fast Mode (Standard)
show_disp(disp_fast, "Fast Mode (LR only)", 1)

# Oben Rechts: Precise Mode LR
show_disp(disp_prec_LR, "Precise Mode (LR Pass)", 2)

# Unten Links: Precise Mode RL (Der Retter für den linken Rand)
show_disp(disp_prec_RL, "Precise Mode (RL Pass - Flipped)", 3)

# Unten Rechts: Echo-Analyse (LRC-Diff)
# Hier siehst du die Fehler am linken Rand leuchten!
show_disp(echo_map, "Echo Analysis (LRC Diff)", 4, cmap="hot")

plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# EPE Check
# ---------------------------------------------------------
# Falls gt_disp.npy nicht existiert, erstellen wir eine Dummy-Maske zum Testen
try:
    gt = np.load("gt_disp.npy")
    gt = torch.tensor(gt, dtype=torch.float32).unsqueeze(0).unsqueeze(0).cuda()
    valid = gt > 0
    
    def calc_epe(pred, gt_val, mask):
        return torch.abs(pred - gt_val)[mask].mean().item()

    epe_fast = calc_epe(disp_fast, gt, valid)
    epe_prec = calc_epe(disp_prec_LR, gt, valid)

    print("-" * 30)
    print(f"EPE Fast Mode   : {epe_fast:.4f} px")
    print(f"EPE Precise Mode: {epe_prec:.4f} px")
    print("-" * 30)
except FileNotFoundError:
    print("GT Datei nicht gefunden. Überspringe EPE Check.")


In [ ]:
import torch
from your_model_file import StereoNetLite_GrabberCore

# 1. Load trained model
model = StereoNetLite_GrabberCore(max_disp=192, num_groups=4, input_size=(480, 640))
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

# 2. Create dummy inputs
dummy_left = torch.randn(1, 1, 480, 640)
dummy_right = torch.randn(1, 1, 480, 640)

# 3. Test forward pass
with torch.no_grad():
    output = model(dummy_left, dummy_right, training=False)
    print(f"Output shape: {output.shape}")  # Should be [1, 1, 480, 640]

# 4. Export to ONNX
torch.onnx.export(
    model,
    (dummy_left, dummy_right),
    "stereo_hailo.onnx",
    export_params=True,
    opset_version=11,
    do_constant_folding=True,
    input_names=['left_image', 'right_image'],
    output_names=['disparity'],
    dynamic_axes={
        'left_image': {0: 'batch_size'},
        'right_image': {0: 'batch_size'},
        'disparity': {0: 'batch_size'}
    }
)

print("✅ ONNX export successful: stereo_hailo.onnx")

# 5. Verify ONNX
import onnx
onnx_model = onnx.load("stereo_hailo.onnx")
onnx.checker.check_model(onnx_model)
print("✅ ONNX model is valid")

# 6. Test ONNX inference
import onnxruntime as ort
session = ort.InferenceSession("stereo_hailo.onnx")
onnx_output = session.run(
    None,
    {'left_image': dummy_left.numpy(), 'right_image': dummy_right.numpy()}
)
print(f"✅ ONNX inference successful, output shape: {onnx_output[0].shape}")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Deine vorhandenen Hilfsfunktionen (PFM & Bilder) ---
def read_pfm(file):
    with open(file, "rb") as f:
        header = f.readline().decode('utf-8').rstrip()
        if header != 'Pf': raise Exception('Keine PFM Pf-Datei.')
        dims = f.readline().decode('utf-8').split()
        width, height = int(dims[0]), int(dims[1])
        scale = float(f.readline().decode('utf-8').rstrip())
        endian = '<' if scale < 0 else '>'
        data = np.fromfile(f, endian + 'f')
        data = np.reshape(data, (height, width))
        data = np.flipud(data)
        data[data == np.inf] = 0
        return data.copy()

# --- 2. Die Analyse-Funktion ---
import torch
import torch.nn.functional as F
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

def analyze_checkpoint_pil(checkpoint_path, left_path, right_path, gt_path, row_y=120):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    target_size = (640, 480) # (W, H)
    
    # 1. Modell laden (weights_only=True für Sicherheit)
    model = StereoFusionAllMode().to(device)
    state_dict = torch.load(checkpoint_path, map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    model.eval()

    # 2. Bilder laden & Resizen (Dein Code)
    left_img_pil = Image.open(left_path).convert('L').resize(target_size, Image.BILINEAR)
    right_img_pil = Image.open(right_path).convert('L').resize(target_size, Image.BILINEAR)
    
    # In Tensor umwandeln [1, 1, 480, 640]
    img_l = torch.from_numpy(np.array(left_img_pil)).float().unsqueeze(0).unsqueeze(0).to(device)
    img_r = torch.from_numpy(np.array(right_img_pil)).float().unsqueeze(0).unsqueeze(0).to(device)

    # 3. Ground Truth laden & Resizen
    gt_orig = read_pfm(gt_path) # Nutzt deine Funktion
    orig_h, orig_w = gt_orig.shape
    
    # WICHTIG: Wenn wir das Bild verkleinern, müssen wir die Disparitätswerte skalieren!
    scale_factor = target_size[0] / orig_w
    gt_rescaled = F.interpolate(torch.from_numpy(gt_orig).unsqueeze(0).unsqueeze(0), 
                                size=(target_size[1], target_size[0]), 
                                mode='nearest').squeeze().numpy()
    gt_rescaled = gt_rescaled * scale_factor # Werte an neue Auflösung anpassen

    # 4. Inferenz
    with torch.no_grad():
        outputs = model(img_l, img_r, mode="precise")
        pred_lr = outputs["LR"][0].cpu().squeeze().numpy()

    # 5. Plotten
    plt.figure(figsize=(15, 6))
    x = np.arange(target_size[0])
    
    plt.plot(x, gt_rescaled[row_y, :], color='black', label='GT (PFM, skaliert)', linewidth=2)
    plt.plot(x, pred_lr[row_y, :], color='red', label='Prediction LR', alpha=0.8)
    
    plt.fill_between(x, gt_rescaled[row_y, :], pred_lr[row_y, :], 
                     where=(np.abs(pred_lr[row_y, :] - gt_rescaled[row_y, :]) > 3),
                     color='red', alpha=0.1, label='Fehler > 3px')

    plt.title(f"Profil-Check Zeile {row_y} (Skalierung: {orig_w} -> {target_size[0]})")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# Aufruf

# ================================================================
# --- 3. DER FUNKTIONSAUFRUF (HIER PASSIERT ES) ---
# ================================================================

if __name__ == "__main__":
    # Pfade anpassen!
    MY_CHECKPOINT = "FusedBackbone-Stereo_5.pth"
    TEST_L = "/home/slarc/datasets/sceneflow/left/0000052.png"
    TEST_R = "/home/slarc/datasets/sceneflow/right/0000052.png"
    TEST_GT = "/home/slarc/datasets/sceneflow/disp/0000052.pfm"

    # Aufruf für die Problem-Zone (Stuhlbein-Echo oben links)
    analyze_checkpoint(
        checkpoint_path=MY_CHECKPOINT,
        left_path=TEST_L,
        right_path=TEST_R,
        gt_path=TEST_GT,
        row_y=120  # Wähle die Zeile, in der das Stuhlbein im Bild sitzt
    )

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import os
path = '/mnt/c/temp/sceneflow/disp/' # Update this
print(f"Directory exists: {os.path.exists(path)}")
print(f"Files in directory: {os.listdir(path)[:5]}") # Shows first 5 files

# 1. Function Call
# Replace 'path_to_your_file.pfm' with your actual file path
file_path = '/mnt/c/temp/FlyingThings3D_subset_disparity.tar/FlyingThings3D_subset_disparity/FlyingThings3D_subset/val/disparity/right/0001000.pfm'
try:
    disparity_map = read_pfm(file_path)
    
    # 2. Visualization
    plt.figure(figsize=(12, 6))
    
    # Use 'magma' or 'plasma' for depth/disparity; it's easier on the eyes
    img = plt.imshow(disparity_map, cmap='magma')
    
    plt.title(f"Stereo Ground Truth Disparity\nResolution: {disparity_map.shape[1]}x{disparity_map.shape[0]}")
    plt.colorbar(img, label='Disparity (pixels)')
    plt.axis('off') # Hide axes for a cleaner look
    
    plt.show()
except FileNotFoundError:
    print(f"Error: The file at {file_path} was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

file_path = '/mnt/c/temp/FlyingThings3D_subset_disparity.tar/FlyingThings3D_subset_disparity/FlyingThings3D_subset/val/disparity/left/0001000.pfm'
try:
    disparity_map = read_pfm(file_path)
    
    # 2. Visualization
    plt.figure(figsize=(12, 6))
    
    # Use 'magma' or 'plasma' for depth/disparity; it's easier on the eyes
    img = plt.imshow(disparity_map, cmap='magma')
    
    plt.title(f"Stereo Ground Truth Disparity\nResolution: {disparity_map.shape[1]}x{disparity_map.shape[0]}")
    plt.colorbar(img, label='Disparity (pixels)')
    plt.axis('off') # Hide axes for a cleaner look
    
    plt.show()

except FileNotFoundError:
    print(f"Error: The file at {file_path} was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
import os
path = '/mnt/c/temp/sceneflow/disp/' # Update this
print(f"Directory exists: {os.path.exists(path)}")
print(f"Files in directory: {os.listdir(path)[:5]}") # Shows first 5 files